# ۵ · پایپ‌لاین کامل — همه با هم

نسخه‌ی اصلاح‌شده‌ی `final_mardani.ipynb`. چهار مسیر جستجو روی یک ایندکس
واحد، با تمام باگ‌های شناسایی‌شده رفع‌شده.

---

## 🗺️ معماری

```
┌─── فاز ایندکس (یک بار به‌ازای هر عکس) ───────────────────┐
│                                                          │
│  تصویر ──> ۱. چهره   (InsightFace)  ──> امبدینگ ۵۱۲ بعدی │
│         ──> ۲. حیوان (MegaDetector) ──> جعبه‌ها          │
│         ──> ۳. غذا   (RT-DETR-X)    ──> پرچم            │
│         ──> ۴. CLIP  (ViT-L/14)     ──> امبدینگ ۷۶۸ بعدی │
│                          ↓                               │
│                  SQLite: پرچم‌ها + بردارها               │
└──────────────────────────────────────────────────────────┘

┌─── فاز پرس‌وجو ──────────────────────────────────────────┐
│  پیش‌فیلتر SQL  ──>  جستجوی برداری فقط روی کاندیداها     │
└──────────────────────────────────────────────────────────┘
```

**ترتیب `چهره → حیوان → غذا` تصادفی نیست.** مرحله غذا برای رد کردن
کادرهایی که روی صورت هستند به جعبه‌های چهره نیاز دارد. نسخه اصلی
`حیوان → غذا → چهره` بود، پس آن فیلتر **هرگز اجرا نشد**.

---

## 💡 ایده‌ی مرکزی: پیش‌فیلتر متادیتا

مهم‌ترین تصمیم معماری کل پروژه. هر تصویر سه پرچم بولی می‌گیرد:

```sql
SELECT file_name, file_path FROM gallery_meta WHERE has_animal = 1;
```

پس جستجوی گونه هرگز به عکسی که حیوان ندارد دست نمی‌زند.

و چون پرچم‌ها از یابنده‌های **کلاس‌ناوابسته** می‌آیند، عوض‌کردن بازشناس
هیچ ایندکس مجددی لازم ندارد — دقیقاً به همین دلیل توانستیم مسیر حیوان
را از MobileNetV3 به CLIP ببریم بدون دست‌زدن به ایندکس.

---

## 🐞 جدول کامل باگ‌ها

| # | ماژول | باگ | راه‌حل |
|:--|:--|:--|:--|
| ۱ | حیوان | نگاشت گونه خراب — فیل = حشره | نگاشت حذف شد، CLIP جایگزین |
| ۲ | غذا | فیلتر تداخل با صورت هرگز اجرا نشد | ترتیب چهره → حیوان → غذا |
| ۳ | حیوان | بدون آستانه اطمینان — ۲۱.۵٪ = تطابق | پسین رقابتی ≥ ۰.۵۵ |
| ۴ | غذا | لیست کلاس ایندکس ≠ جستجو (صندلی) | یک منبع + قاعده زیرمجموعه |
| ۵ | حیوان | `"animal"` همه چیز را برمی‌گرداند | نگاشت حذف شد |
| ۶ | چهره | عکس دوچهره دو بار در نتایج | `best_per_image` |
| ۷ | همه | `device == "cuda"` همیشه False | `Runtime` یکپارچه |
| ۸ | داده | `pickle` — ریسک امنیتی | `float32` خام |
| ۹ | داده | بدون کلید اصلی → ردیف تکراری | `PRIMARY KEY` |
| ۱۰ | متن | کسینوس خام ۲۳٪ = شبیه شکست | امتیاز نسبی کالیبره |

In [ ]:
# ── نصب وابستگی‌ها ───────────────────────────────────────────────────
# ترتیب مهم است: insightface نسخه CPU از onnxruntime نصب می‌کند،
# پس اول آن را حذف و بعد نسخه GPU را نصب می‌کنیم.
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q insightface ultralytics transformers opencv-python tqdm
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu

import onnxruntime as ort
print("✅ نصب کامل شد")
print("موتورهای در دسترس:", ort.get_available_providers())

In [ ]:
# ── تشخیص دستگاه و دقت عددی ─────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    کد قبلی سه جور مقایسه داشت:
#        if device.type == "cuda":   ← درست
#        if device == "cuda":        ← همیشه False !
#
#    چون device یک شیء torch.device است، نه رشته.
#    تست شده روی torch 2.8:  torch.device('cuda') == 'cuda'  →  False
#
#    نتیجه: روی GPU مدل half می‌شد ولی ورودی float32 می‌ماند →
#    RuntimeError: expected scalar type Half but found Float
#
# ✅ راه‌حل: دستگاه و dtype را یکجا حل می‌کنیم تا نتوانند با هم اختلاف پیدا کنند.

import torch
from dataclasses import dataclass


@dataclass(frozen=True)
class Runtime:
    device: torch.device
    use_half: bool

    @property
    def is_cuda(self) -> bool:
        return self.device.type == "cuda"

    def cast_inputs(self, inputs: dict) -> dict:
        """انتقال ورودی به دستگاه با dtype هماهنگ.
        فقط تنسورهای اعشاری half می‌شوند؛ input_ids باید عدد صحیح بماند."""
        out = {}
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                v = v.to(self.device)
                if self.use_half and v.is_floating_point():
                    v = v.half()
            out[k] = v
        return out

    def prepare_model(self, model):
        model = model.to(self.device)
        if self.use_half:
            model = model.half()
        return model.eval()


def resolve_runtime(preference: str = "auto", half: bool = True) -> Runtime:
    name = preference
    if preference == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(name)
    # fp16 روی CPU کندتر از fp32 است و بعضی عملیات پشتیبانی نمی‌شوند
    return Runtime(device=device, use_half=half and device.type == "cuda")


RT = resolve_runtime()
print(f"🚀 دستگاه: {RT.device}   |   نیمه‌دقت (FP16): {RT.use_half}")

In [ ]:
# ── ذخیره‌سازی امبدینگ ──────────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    کد قبلی از pickle استفاده می‌کرد:
#        pickle.dumps(face.normed_embedding.tolist())
#
#    دو مشکل:
#    ۱) امنیتی: pickle.loads روی فایلی که خودت نساخته‌ای =
#       اجرای کد دلخواه. فایل ایندکس دقیقاً همان چیزی است که
#       بین سیستم‌ها کپی می‌شود.
#    ۲) حجم: .tolist() یک لیست پایتون می‌سازد، نه آرایه —
#       چند برابر بایت خام.
#
# ✅ راه‌حل: float32 خام با tobytes/frombuffer

import numpy as np

DTYPE = np.dtype(np.float32).newbyteorder("<")   # little-endian، قابل حمل


def to_blob(vector) -> bytes:
    arr = np.asarray(vector, dtype=np.float32).ravel()
    if arr.size == 0:
        raise ValueError("امبدینگ خالی")
    if not np.all(np.isfinite(arr)):
        # یک NaN تمام شباهت‌هایی که در آن شرکت کند را مسموم می‌کند
        raise ValueError("امبدینگ شامل NaN یا بی‌نهایت است")
    return arr.astype(DTYPE, copy=False).tobytes()


def from_blob(blob: bytes) -> np.ndarray:
    if not blob:
        raise ValueError("بلاب خالی")
    if len(blob) % DTYPE.itemsize:
        raise ValueError("طول بلاب نادرست — شاید ایندکس با نسخه pickle قدیمی ساخته شده")
    return np.frombuffer(blob, dtype=DTYPE).astype(np.float32, copy=True)


# تست سریع
_v = np.random.randn(512).astype(np.float32)
assert np.allclose(from_blob(to_blob(_v)), _v, atol=1e-6)
print(f"✅ ذخیره‌سازی float32 سالم است — یک بردار ۵۱۲ بعدی = {len(to_blob(_v)):,} بایت")

In [ ]:
# ── ابزار جعبه‌ها ───────────────────────────────────────────────────
#
# 🐞 باگی که رفع شد:
#    برش بدون کلمپ مرزی:  image[y1-pad : y2+pad, x1-pad : x2+pad]
#    اگر x1-pad منفی شود، numpy خطا نمی‌دهد — بی‌صدا از ته آرایه
#    برمی‌دارد و برش غلط می‌دهد. یعنی حیوانی که به لبه کادر چسبیده،
#    از روی پیکسل‌های اشتباه طبقه‌بندی می‌شد.

def area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def overlap_fraction(box, other) -> float:
    """چه کسری از box داخل other است.

    عمداً نامتقارن: سؤال «چقدر از این کادرِ غذا روی صورت است»،
    نه «این دو کادر چقدر شبیه‌اند». IoU جواب سؤال اشتباه را می‌دهد —
    کادر کوچک غذا کاملاً داخل کادر بزرگ صورت، IoU پایینی دارد
    ولی overlap_fraction آن ۱.۰ است.
    """
    ax1, ay1, ax2, ay2 = box
    bx1, by1, bx2, by2 = other
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix1 >= ix2 or iy1 >= iy2:
        return 0.0
    a = area(box)
    return ((ix2 - ix1) * (iy2 - iy1) / a) if a else 0.0


def overlaps_any(box, others, threshold: float) -> bool:
    return any(overlap_fraction(box, o) > threshold for o in others)


def pad_box(box, width, height, ratio=0.0, pixels=0):
    """بزرگ‌کردن کادر + کلمپ به مرز تصویر (✅ رفع باگ اسلایس منفی)."""
    x1, y1, x2, y2 = box
    px = int(round((x2 - x1) * ratio)) + pixels
    py = int(round((y2 - y1) * ratio)) + pixels
    return (max(0, x1 - px), max(0, y1 - py),
            min(width, x2 + px), min(height, y2 + py))


def is_large_enough(box, min_px: int) -> bool:
    """رد کردن کادرهای ریز.

    🐞 قبلاً هیچ حداقلی نبود: یک کادر ۱۲×۹ پیکسل ۲۰ برابر بزرگ می‌شد
    به ۲۲۴×۲۲۴ و یک برچسب با درصد بالا می‌گرفت.
    """
    x1, y1, x2, y2 = box
    return (x2 - x1) >= min_px and (y2 - y1) >= min_px


print("✅ ابزار جعبه‌ها آماده (با کلمپ مرزی و حداقل اندازه)")

In [ ]:
# ── امتیازدهی رقابتی ────────────────────────────────────────────────
#
# 💡 بهترین ایده‌ی کل پروژه — و حالا در هر سه مسیر تشخیص استفاده می‌شود.
#
# به‌جای آستانه گذاشتن روی شباهت خام، پرسش را رقابتی می‌کنیم:
#
#     prompts = ["a photo of a zebra",                    ← فرضیه
#                "a photo of a different kind of animal", ← رقیب
#                "a photo of scenery with no animal"]     ← رقیب
#     score = softmax(logits)[0]
#
# چرا بهتر است؟ شباهت کسینوسی «فرضیه صفر» ندارد — هر برشی یک عددی
# می‌دهد و جای برش اصولی ندارد. softmax روی رقبا به مدل اجازه می‌دهد
# بگوید «هیچ‌کدام»، پس خروجی یک احتمال کالیبره است نه یک فاصله.
#
# شاهدش در همین پروژه: مسیر غذا (با رقیب) ۹۶–۹۹٪ می‌داد،
# مسیر متنی (کسینوس خام) برای بهترین نتیجه‌اش ۲۳٪ نشان می‌داد.

import math
from dataclasses import dataclass


def softmax(values):
    if not values:
        return []
    top = max(values)
    exps = [math.exp(v - top) for v in values]   # پایدار عددی
    total = sum(exps)
    return [e / total for e in exps]


def contrastive_score(logits) -> float:
    """احتمال پسین فرضیه در برابر رقبایش."""
    if not logits:
        raise ValueError("logits خالی")
    if len(logits) == 1:
        raise ValueError("حداقل یک prompt رقیب لازم است؛ "
                         "با یک کاندیدا softmax همیشه ۱.۰ است")
    return softmax(logits)[0]


@dataclass(frozen=True)
class Match:
    file_name: str
    file_path: str
    score: float
    box: tuple = None
    label: str = None


def best_per_image(matches):
    """یک نتیجه به‌ازای هر تصویر — قوی‌ترین نمونه.

    🐞 باگی که رفع شد:
       نسخه یکپارچه break هر تصویر را از دست داده بود، پس عکسی با
       دو چهره‌ی منطبق دو بار در نتایج ظاهر می‌شد.

       ضمناً کد قبلی «اولین» نمونه را ثبت می‌کرد و همان را کلید
       مرتب‌سازی می‌کرد — پس عکسی که گربه دومش ۹۵٪ بود با ۲۶٪
       رتبه‌بندی می‌شد.
    """
    best = {}
    for m in matches:
        cur = best.get(m.file_name)
        if cur is None or m.score > cur.score:
            best[m.file_name] = m
    return sorted(best.values(), key=lambda m: m.score, reverse=True)


assert abs(sum(softmax([1.0, 2.0, 3.0])) - 1.0) < 1e-9
assert contrastive_score([10.0, 0.0, 0.0]) > 0.95
assert contrastive_score([0.0, 10.0, 0.0]) < 0.05
print("✅ امتیازدهی رقابتی آماده")

## ⚙️ تنظیمات یکپارچه

همه‌ی ثابت‌های هر چهار مسیر، یکجا، با اعتبارسنجی.

In [ ]:
from dataclasses import dataclass

COCO = {39:"bottle",40:"wine glass",41:"cup",42:"fork",43:"knife",44:"spoon",
        45:"bowl",46:"banana",47:"apple",48:"sandwich",49:"orange",50:"broccoli",
        51:"carrot",52:"hot dog",53:"pizza",54:"donut",55:"cake",56:"chair"}

@dataclass(frozen=True)
class Config:
    gallery: str = "/content/gallery"
    database: str = "unified_index.db"

    # ── چهره ──
    face_threshold: float = 0.46
    det_size_large: int = 640
    det_size_small: int = 320
    small_image_px: int = 400

    # ── حیوان ──
    animal_class_id: int = 0            # تنسور خام YOLOv5: ۰=حیوان
    animal_gate_conf: float = 0.30      # ایندکس (سخاوتمند)
    animal_search_conf: float = 0.40    # جستجو (سخت‌گیر)
    animal_classifier_top_k: int = 5        # حدس‌های ذخیره‌شده در ایندکس
    animal_classifier_threshold: float = 0.25
    animal_clip_threshold: float = 0.55     # فقط گونه‌های خارج از واژگان
    animal_padding_ratio: float = 0.15
    animal_min_px: int = 40

    # ── غذا ──
    food_gate_classes: tuple = tuple(range(39, 56))    # ۵۶ (صندلی) عمداً نیست
    food_search_classes: tuple = tuple(range(45, 56))
    food_gate_conf: float = 0.15
    food_search_conf: float = 0.25
    food_threshold: float = 0.60
    food_padding_px: int = 20
    food_min_px: int = 50
    face_overlap_reject: float = 0.50

    # ── متن ──
    top_k: int = 5

    max_index_dimension: int = 1600
    workers: int = 8
    checkpoint_every: int = 50

    def animal_prompts(self, q):
        return [f"a photo of a {q}",
                "a photo of a different kind of animal",
                "a photo of scenery with no animal in it"]

    def food_prompts(self, q):
        return [q,
                "a photo of a completely different food",
                "a photo of an empty plate or background"]

    def validate(self):
        """قواعدی که یک کلاس کامل از باگ‌ها را غیرممکن می‌کنند."""
        errs = []
        # گیت فیلتر سخت است: چیزی که رد کند، جستجو هرگز نمی‌بیند
        if self.animal_gate_conf > self.animal_search_conf:
            errs.append("گیت حیوان سخت‌گیرتر از جستجوست")
        if self.food_gate_conf > self.food_search_conf:
            errs.append("گیت غذا سخت‌گیرتر از جستجوست")
        stray = set(self.food_search_classes) - set(self.food_gate_classes)
        if stray:
            errs.append(f"کلاس‌های {sorted(stray)} جستجو می‌شوند ولی ایندکس نشده‌اند")
        if 56 in self.food_gate_classes or 56 in self.food_search_classes:
            errs.append("کلاس ۵۶ صندلی است، غذا نیست")
        if errs:
            raise ValueError("تنظیمات نامعتبر:\n  - " + "\n  - ".join(errs))
        return True

CFG = Config()
CFG.validate()
print("✅ تنظیمات معتبر")
print("   حیوان: گیت", CFG.animal_gate_conf, "≤ جستجو", CFG.animal_search_conf)
print("   غذا  : جستجو زیرمجموعه‌ی گیت ✓  |  صندلی وارد نشده ✓")

## 🧠 بارگذاری همه‌ی مدل‌ها

یک بار بارگذاری، ماندگار در VRAM.

In [ ]:
import os
import urllib.request

import torch
from insightface.app import FaceAnalysis
from transformers import CLIPModel, CLIPProcessor
from ultralytics import RTDETR

# ── ۱. CLIP: قلب سه مسیر از چهار مسیر ──
CLIP_NAME = "openai/clip-vit-large-patch14"
clip_model = RT.prepare_model(CLIPModel.from_pretrained(CLIP_NAME))
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)
print("✅ CLIP ViT-L/14 —", "FP16" if RT.use_half else "FP32")

# ── ۲. چهره ──
providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
             if RT.is_cuda else ["CPUExecutionProvider"])
face_app = FaceAnalysis(name="buffalo_l", providers=providers)
face_app.prepare(ctx_id=0 if RT.is_cuda else -1,
                 det_size=(CFG.det_size_large, CFG.det_size_large))
print("✅ InsightFace buffalo_l")

# ── ۳. حیوان (yolov5 pin شده) ──
MD_PATH = "md_v5a.0.0.pt"
if not os.path.exists(MD_PATH):
    urllib.request.urlretrieve(
        "https://github.com/ecologize/CameraTraps/releases/download/v5.0/md_v5a.0.0.pt",
        MD_PATH)
animal_detector = torch.hub.load("ultralytics/yolov5:v7.0", "custom",
                                 path=MD_PATH, trust_repo=True).to(RT.device)
print("✅ MegaDetector v5a (yolov5 @ v7.0)")

# ── ۴. طبقه‌بند گونه: MobileNetV3-Large ──
# مسیر اصلی شناسایی. ۳۷۳ برابر ارزان‌تر از CLIP به‌ازای هر برش
# (۰.۲۱۷ در برابر ~۸۱ GFLOPs) با دقت ImageNet قابل مقایسه.
from torchvision import models, transforms

_w = models.MobileNet_V3_Large_Weights.DEFAULT
classifier = models.mobilenet_v3_large(weights=_w).to(RT.device).eval()
classes = _w.meta["categories"]
print(f"✅ MobileNetV3-Large — {_w.meta['_metrics']['ImageNet-1K']['acc@1']}% top-1")

# ✅ زنجیره درست: Resize(256)+CenterCrop نسبت تصویر را حفظ می‌کند.
#    نسخه اصلی Resize((224,224)) داشت که برش‌های غیرمربع را له می‌کرد.
preprocess = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

# ── ۵. غذا ──
food_detector = RTDETR("rtdetr-x.pt")
print("✅ RT-DETR-X")

## 🗄️ پایگاه داده

کلیدهای اصلی روی همه‌ی جداول، `float32` به‌جای pickle، و `content_hash`
برای تشخیص فایل ویرایش‌شده.

In [ ]:
import hashlib
import sqlite3
from contextlib import contextmanager
from pathlib import Path

SCHEMA = """
CREATE TABLE IF NOT EXISTS gallery_meta (
    file_name    TEXT PRIMARY KEY,
    file_path    TEXT NOT NULL,
    content_hash TEXT NOT NULL,
    has_face     INTEGER NOT NULL DEFAULT 0,
    has_animal   INTEGER NOT NULL DEFAULT 0,
    has_food     INTEGER NOT NULL DEFAULT 0
);
CREATE TABLE IF NOT EXISTS face_embeddings (
    file_name TEXT NOT NULL, bbox TEXT NOT NULL, embedding BLOB NOT NULL,
    PRIMARY KEY (file_name, bbox),
    FOREIGN KEY (file_name) REFERENCES gallery_meta(file_name) ON DELETE CASCADE
);
CREATE TABLE IF NOT EXISTS clip_embeddings (
    file_name TEXT PRIMARY KEY, embedding BLOB NOT NULL,
    FOREIGN KEY (file_name) REFERENCES gallery_meta(file_name) ON DELETE CASCADE
);
CREATE TABLE IF NOT EXISTS animal_boxes (
    file_name TEXT NOT NULL, bbox TEXT NOT NULL, confidence REAL NOT NULL,
    PRIMARY KEY (file_name, bbox),
    FOREIGN KEY (file_name) REFERENCES gallery_meta(file_name) ON DELETE CASCADE
);
-- طبقه‌بندی در ایندکس ذخیره می‌شود تا پرس‌وجو هیچ استنتاجی نکند
CREATE TABLE IF NOT EXISTS animal_predictions (
    file_name TEXT NOT NULL, bbox TEXT NOT NULL, rank INTEGER NOT NULL,
    class_id INTEGER NOT NULL, probability REAL NOT NULL,
    PRIMARY KEY (file_name, bbox, rank)
);
CREATE INDEX IF NOT EXISTS idx_class ON animal_predictions(class_id);
CREATE INDEX IF NOT EXISTS idx_face   ON gallery_meta(has_face);
CREATE INDEX IF NOT EXISTS idx_animal ON gallery_meta(has_animal);
CREATE INDEX IF NOT EXISTS idx_food   ON gallery_meta(has_food);
"""

@contextmanager
def connect():
    conn = sqlite3.connect(CFG.database)
    try:
        conn.execute("PRAGMA foreign_keys = ON")
        yield conn
        conn.commit()
    finally:
        conn.close()

def content_hash(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

with connect() as c:
    c.executescript(SCHEMA)
print("✅ پایگاه داده آماده:", CFG.database)

## 📥 ایندکس‌گذاری

**ترتیب مراحل اینجا حیاتی است.** چهره اول، چون مرحله غذا به جعبه‌های
آن نیاز دارد.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def embed_image(img_bgr):
    """امبدینگ CLIP تصویر — نرمال‌شده تا ضرب داخلی = کسینوس."""
    pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    inputs = RT.cast_inputs(dict(clip_processor(images=pil, return_tensors="pt")))
    with torch.no_grad():
        feats = clip_model.get_image_features(**inputs)
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats.float().cpu().numpy().ravel()


def detect_animals(img_rgb, conf):
    with torch.no_grad():
        preds = animal_detector(img_rgb).xyxy[0].cpu().numpy()
    return [((int(r[0]), int(r[1]), int(r[2]), int(r[3])), float(r[4]))
            for r in preds
            if int(r[5]) == CFG.animal_class_id and float(r[4]) >= conf]


def detect_food(path_or_img, classes, conf):
    res = food_detector(path_or_img, classes=list(classes), conf=conf, verbose=False)
    out = []
    for r in res:
        for b in r.boxes:
            x1, y1, x2, y2 = (int(v) for v in b.xyxy[0])
            out.append(((x1, y1, x2, y2), float(b.conf[0])))
    return out


def load_and_resize(path):
    try:
        img = cv2.imread(str(path))
        if img is None:
            return path, None
        h, w = img.shape[:2]
        if max(h, w) > CFG.max_index_dimension:
            s = CFG.max_index_dimension / max(h, w)
            img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
        return path, img
    except Exception:
        return path, None


def build_index():
    paths = sorted(p for p in Path(CFG.gallery).iterdir()
                   if p.suffix.lower() in IMAGE_SUFFIXES)     # حساس به حروف نیست
    stats = {"indexed": 0, "skipped": 0, "failed": 0, "food_rejected": 0}

    with connect() as conn:
        pending = []
        for p in paths:
            h = content_hash(p)
            row = conn.execute("SELECT content_hash FROM gallery_meta WHERE file_name=?",
                               (p.name,)).fetchone()
            if row is None or row[0] != h:
                pending.append((p, h))
            else:
                stats["skipped"] += 1

        if not pending:
            print("✅ ایندکس به‌روز است —", stats)
            return stats

        print(f"⏳ ایندکس {len(pending)} تصویر (از {len(paths)})")
        hashes = dict(pending)

        with ThreadPoolExecutor(max_workers=CFG.workers) as pool:
            loaded = pool.map(load_and_resize, [p for p, _ in pending])

            for path, img in tqdm(loaded, total=len(pending), desc="Indexing"):
                if img is None:
                    stats["failed"] += 1
                    continue
                try:
                    h_img, w_img = img.shape[:2]
                    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                    # ═══ ۱. چهره — اول، چون غذا به آن نیاز دارد ═══
                    faces = face_app.get(img)
                    face_boxes = [tuple(int(v) for v in f.bbox[:4]) for f in faces]

                    # ═══ ۲. حیوان — گیت سخاوتمند ═══
                    animals = [(b, c) for b, c in detect_animals(rgb, CFG.animal_gate_conf)
                               if is_large_enough(b, CFG.animal_min_px)]

                    # ═══ ۳. غذا — حالا face_boxes موجود است ═══
                    foods = []
                    for box, conf in detect_food(str(path), CFG.food_gate_classes,
                                                 CFG.food_gate_conf):
                        if not is_large_enough(box, CFG.food_min_px):
                            continue
                        if overlaps_any(box, face_boxes, CFG.face_overlap_reject):
                            stats["food_rejected"] += 1     # فیلتری که قبلاً اجرا نمی‌شد
                            continue
                        foods.append(box)

                    # ═══ ۴. نوشتن ═══
                    for t in ("face_embeddings","clip_embeddings","animal_boxes",
                              "animal_predictions"):
                        conn.execute(f"DELETE FROM {t} WHERE file_name=?", (path.name,))
                    conn.execute(
                        "INSERT INTO gallery_meta VALUES (?,?,?,?,?,?) "
                        "ON CONFLICT(file_name) DO UPDATE SET "
                        "file_path=excluded.file_path, content_hash=excluded.content_hash, "
                        "has_face=excluded.has_face, has_animal=excluded.has_animal, "
                        "has_food=excluded.has_food",
                        (path.name, str(path), hashes[path],
                         int(bool(faces)), int(bool(animals)), int(bool(foods))))

                    for f in faces:
                        box = ",".join(str(int(v)) for v in f.bbox[:4])
                        conn.execute("INSERT OR REPLACE INTO face_embeddings VALUES (?,?,?)",
                                     (path.name, box, to_blob(f.normed_embedding)))
                    # ⚡ طبقه‌بندی همین‌جا، نه در زمان پرس‌وجو
                    valid = []
                    for b, c in animals:
                        pb = pad_box(b, w_img, h_img, ratio=CFG.animal_padding_ratio)
                        patch = rgb[pb[1]:pb[3], pb[0]:pb[2]]
                        if patch.size:
                            valid.append((b, c, patch))
                    if valid:
                        batch = torch.stack([preprocess(cr) for _, _, cr in valid])
                        with torch.no_grad():
                            probs = torch.softmax(classifier(batch.to(RT.device)), dim=1)
                        top = torch.topk(probs, k=CFG.animal_classifier_top_k, dim=1)
                        for (b, c, _), idxs, vals in zip(valid, top.indices, top.values):
                            key = ",".join(map(str, b))
                            conn.execute("INSERT OR REPLACE INTO animal_boxes VALUES (?,?,?)", (path.name, key, c))
                            for r_, (cid, pr) in enumerate(zip(idxs.tolist(),
                                                               vals.tolist())):
                                conn.execute(
                                    "INSERT OR REPLACE INTO animal_predictions VALUES (?,?,?,?,?)",
                                    (path.name, key, r_, cid, pr))
                    conn.execute("INSERT OR REPLACE INTO clip_embeddings VALUES (?,?)",
                                 (path.name, to_blob(embed_image(img))))

                    stats["indexed"] += 1
                    if stats["indexed"] % CFG.checkpoint_every == 0:
                        conn.commit()                        # checkpoint
                except Exception as e:
                    print("⚠️ ", path.name, ":", e)          # نه except خالی
                    stats["failed"] += 1

    print("✅ تمام شد —", stats)
    print("   ", stats["food_rejected"], "کادر غذا روی صورت رد شد")
    return stats

# build_index()

## 🔎 چهار مسیر جستجو

In [ ]:
import time


def score_crops(crops, prompts):
    inputs = clip_processor(text=list(prompts), images=list(crops),
                            return_tensors="pt", padding=True)
    inputs = RT.cast_inputs(dict(inputs))
    with torch.no_grad():
        return clip_model(**inputs).logits_per_image.float().cpu().tolist()


# ═══════════════ ۱. چهره ═══════════════
def search_face(query_image):
    img = cv2.imread(str(query_image))
    if img is None:
        raise ValueError("تصویر خوانده نشد")
    h, w = img.shape[:2]
    size = CFG.det_size_large if max(h, w) > CFG.small_image_px else CFG.det_size_small
    face_app.prepare(ctx_id=0 if RT.is_cuda else -1, det_size=(size, size))
    faces = face_app.get(img)
    if not faces:
        face_app.prepare(ctx_id=0 if RT.is_cuda else -1,
                         det_size=(CFG.det_size_large, CFG.det_size_large))
        faces = face_app.get(img)
    if not faces:
        raise ValueError("چهره‌ای در عکس پرس‌وجو نبود")

    q = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1])).normed_embedding
    q = np.asarray(q, dtype=np.float32).ravel()

    t0 = time.time()
    matches = []
    with connect() as conn:
        for name, bbox, blob, path in conn.execute(
                "SELECT f.file_name,f.bbox,f.embedding,g.file_path "
                "FROM face_embeddings f JOIN gallery_meta g USING(file_name)"):
            emb = from_blob(blob)
            if emb.shape != q.shape:
                raise ValueError(f"ناسازگاری ابعاد در {name}")
            sim = float(np.dot(q, emb))
            if sim >= CFG.face_threshold:
                matches.append(Match(name, path, sim,
                                     box=tuple(int(v) for v in bbox.split(","))))
    return _report(best_per_image(matches), t0)


# ═══════════════ ۲. حیوان (ترکیبی) ═══════════════
#
# پرس‌وجو به یکی از دو موتور مسیریابی می‌شود:
#   داخل واژگان ImageNet → MobileNetV3 از ایندکس (صفر استنتاج)
#   خارج از واژگان       → CLIP روی برش‌ها (تنها گزینه، ۳۷۳× گران‌تر)
import re

GENERIC_TERMS = {
    "dog": ("terrier","retriever","hound","spaniel","collie","poodle","pug",
            "bulldog","husky","malamute","chihuahua","beagle","rottweiler",
            "schnauzer","setter","pointer","sheepdog","mastiff","pinscher",
            "shepherd","boxer"),
    "cat": ("tabby","Persian cat","Siamese cat","Egyptian cat"),
    "big cat": ("lion","tiger","leopard","snow leopard","jaguar","cheetah",
                "cougar","lynx"),
    "elephant": ("African elephant","Indian elephant","tusker"),
    "bear": ("brown bear","American black bear","ice bear","sloth bear"),
    "monkey": ("macaque","langur","baboon","guenon","colobus","marmoset","capuchin"),
    "ape": ("gorilla","chimpanzee","orangutan","gibbon","siamang"),
}
IRREGULAR = {"wolves":"wolf","geese":"goose","mice":"mouse","oxen":"ox",
             "sheep":"sheep","deer":"deer","fish":"fish","foxes":"fox",
             "butterflies":"butterfly"}


def normalise(q):
    q = " ".join(q.lower().split())
    if q in IRREGULAR:
        return IRREGULAR[q]
    if q.endswith("ies") and len(q) > 4:
        return q[:-3] + "y"
    if q.endswith("es") and len(q) > 4 and q[-3] in "sxzh":
        return q[:-2]
    if q.endswith("s") and not q.endswith("ss") and len(q) > 3:
        return q[:-1]
    return q


def word_match(term, name):
    """مرز کلمه — همان چیزی که باگ فیل را رفع می‌کند.

    're.search(r"\bant\b", "African elephant")' غلط است،
    در حالی که '"ant" in "African elephant"' درست بود.
    """
    pat = r"" + re.escape(term) + r""
    return any(re.search(pat, part.strip(), re.I) for part in name.split(","))


def exact_name(term, name):
    """آیا term دقیقاً یکی از نام‌های این کلاس است؟

    لازم است چون 'lion' یک کلمه کامل داخل 'sea lion' هم هست.
    """
    return any(term == part.strip().lower() for part in name.split(","))


def resolve(query):
    """پرس‌وجو → اندیس‌های ImageNet. لیست تهی یعنی: برو سراغ CLIP."""
    n = normalise(query)

    ids = [i for i, c in enumerate(classes) if exact_name(n, c)]
    if ids:
        return ids, "exact"

    if n in GENERIC_TERMS:
        s = set()
        for term in GENERIC_TERMS[n]:
            s.update(i for i, c in enumerate(classes) if word_match(term, c))
        s.update(i for i, c in enumerate(classes) if word_match(n, c))
        if s:
            return sorted(s), "generic"

    ids = [i for i, c in enumerate(classes) if word_match(n, c)]
    if ids:
        return ids, "partial"

    return [], "unresolved"


def search_animal(query):
    t0 = time.time()
    ids, via = resolve(query)

    # ⚡ مسیر ارزان: فقط SQL، هیچ مدلی اجرا نمی‌شود
    if ids:
        print(f"🔀 '{query}' → {len(ids)} کلاس ImageNet ({via}) → MobileNetV3 از ایندکس")
        ph = ",".join("?" * len(ids))
        with connect() as conn:
            rows = conn.execute(
                "SELECT p.file_name, g.file_path, p.bbox, p.class_id, p.probability "
                "FROM animal_predictions p JOIN gallery_meta g USING(file_name) "
                f"WHERE p.class_id IN ({ph}) AND p.probability >= ?",
                (*ids, CFG.animal_classifier_threshold)).fetchall()
        matches = [Match(n, p, pr, box=tuple(int(v) for v in bb.split(",")),
                         label=classes[cid])
                   for n, p, bb, cid, pr in rows]
        return _report(best_per_image(matches), t0)

    # 🐢 مسیر گران: تنها راه برای گونه‌ای که در واژگان نیست
    print(f"🔀 '{query}' خارج از واژگان ImageNet → CLIP روی برش‌ها")
    with connect() as conn:
        rows = conn.execute("SELECT file_name,file_path FROM gallery_meta "
                            "WHERE has_animal=1").fetchall()
        stored = {}
        for n_, b_, c_ in conn.execute(
                "SELECT file_name,bbox,confidence FROM animal_boxes"):
            stored.setdefault(n_, []).append(
                (tuple(int(v) for v in b_.split(",")), c_))

    cands, crops = [], []
    for name, path in rows:
        img = cv2.imread(path)
        if img is None:
            continue
        h, w = img.shape[:2]
        for box, conf in stored.get(name, []):
            if conf < CFG.animal_search_conf:
                continue
            pb = pad_box(box, w, h, ratio=CFG.animal_padding_ratio)
            patch = img[pb[1]:pb[3], pb[0]:pb[2]]
            if patch.size:
                cands.append((name, path, box))
                crops.append(Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)))

    if not crops:
        return _report([], t0)
    logits = score_crops(crops, CFG.animal_prompts(query))
    matches = [Match(n, p, contrastive_score(r), box=b, label=query)
               for (n, p, b), r in zip(cands, logits)
               if contrastive_score(r) >= CFG.animal_clip_threshold]
    return _report(best_per_image(matches), t0)


# ═══════════════ ۳. غذا ═══════════════
def search_food(query):
    t0 = time.time()
    with connect() as conn:
        rows = conn.execute("SELECT file_name,file_path FROM gallery_meta "
                            "WHERE has_food=1").fetchall()
    print(f"پیش‌فیلتر: {len(rows)} کاندیدا")

    cands, crops = [], []
    for name, path in rows:
        img = cv2.imread(path)
        if img is None:
            continue
        h, w = img.shape[:2]
        for box, conf in detect_food(path, CFG.food_search_classes, CFG.food_search_conf):
            if not is_large_enough(box, CFG.food_min_px):
                continue
            pb = pad_box(box, w, h, pixels=CFG.food_padding_px)
            patch = img[pb[1]:pb[3], pb[0]:pb[2]]
            if patch.size:
                cands.append((name, path, box))
                crops.append(Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)))

    if not crops:
        return _report([], t0)
    logits = score_crops(crops, CFG.food_prompts(query))
    matches = [Match(n, p, contrastive_score(r), box=b, label=query)
               for (n, p, b), r in zip(cands, logits)
               if contrastive_score(r) >= CFG.food_threshold]
    return _report(best_per_image(matches), t0)


# ═══════════════ ۴. متن آزاد ═══════════════
def search_text(query, top_k=None):
    top_k = CFG.top_k if top_k is None else top_k
    t0 = time.time()

    inputs = RT.cast_inputs(dict(clip_processor(text=[query], return_tensors="pt",
                                                padding=True)))
    with torch.no_grad():
        feats = clip_model.get_text_features(**inputs)
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    q = feats.float().cpu().numpy().ravel()

    scored = []
    with connect() as conn:
        for name, blob, path in conn.execute(
                "SELECT c.file_name,c.embedding,g.file_path "
                "FROM clip_embeddings c JOIN gallery_meta g USING(file_name)"):
            emb = from_blob(blob)
            if emb.shape != q.shape:
                raise ValueError(f"ناسازگاری ابعاد در {name}")
            scored.append(Match(name, path, float(np.dot(q, emb))))

    scored.sort(key=lambda m: m.score, reverse=True)
    top = scored[:top_k]

    # ✅ رفع باگ ۱۰: کسینوس خام CLIP در باند ۰.۱۵–۰.۳۵ است، پس ۲۳٪
    # شبیه شکست به نظر می‌رسد. با دمای خود CLIP مقیاس می‌کنیم تا
    # «چقدر نتیجه اول از بقیه بهتر است» خوانا شود.
    rel = softmax([m.score * 100.0 for m in top])
    print(f"جستجو در {time.time()-t0:.4f} ثانیه")
    for i, (m, r) in enumerate(zip(top, rel), 1):
        print(f" [{i:>2}] {m.file_name:<28} کسینوس {m.score:.4f}  |  نسبی {r*100:5.1f}%")
    return top


def _report(results, t0):
    print(f"⚡ {time.time()-t0:.4f} ثانیه — {len(results)} نتیجه")
    for i, m in enumerate(results, 1):
        print(f" [{i:>2}] {m.file_name:<28} {m.score*100:6.2f}%")
    return results

## 🎛️ منوی تعاملی

In [ ]:
def menu():
    print("\n" + "="*56)
    print("  سیستم جستجوی هوشمند گالری")
    print("="*56)
    print("  ۱. 👤 جستجوی چهره (با عکس)")
    print("  ۲. 🦁 جستجوی گونه جانوری (واژگان باز)")
    print("  ۳. 🍱 جستجوی غذا")
    print("  ۴. 🌐 جستجوی متنی آزاد")
    print("  ۵. 📥 ساخت/به‌روزرسانی ایندکس")
    print("="*56)

    choice = input("گزینه (۱ تا ۵): ").strip()
    try:
        if choice == "1":
            return search_face(input("مسیر عکس چهره: ").strip())
        if choice == "2":
            return search_animal(input("نام حیوان (مثلا red panda): ").strip())
        if choice == "3":
            return search_food(input("نام غذا (مثلا French fries): ").strip())
        if choice == "4":
            return search_text(input("توصیف: ").strip())
        if choice == "5":
            return build_index()
        print("گزینه نامعتبر")
    except (ValueError, FileNotFoundError) as e:
        print("خطا:", e)      # پیام واضح، نه کرش

# menu()

## 📋 خلاصه‌ی نهایی

### آنچه نگه داشتیم (تصمیم‌های درست نسخه اصلی)

| تصمیم | چرا درست بود |
|:--|:--|
| **پیش‌فیلتر متادیتا** | مهم‌ترین ایده‌ی معماری — پرس‌وجو فقط روی کاندیداها |
| **پرامپت‌های رقیب CLIP** | فرضیه صفر می‌سازد؛ خروجی کالیبره به‌جای فاصله |
| **معماری دومرحله‌ای** | یابنده جعبه می‌دهد و مقیاس را حفظ می‌کند |
| **`normed_embedding`** | ضرب داخلی دقیقاً کسینوس می‌شود |
| **پدینگ درصدی + کلمپ** | زمینه متناسب با اندازه سوژه |
| **ایندکس افزایشی** | فکر محصولی |

### آنچه عوض شد

| | قبلاً | الان |
|:--|:--|:--|
| بازشناس حیوان | MobileNetV3-Small + ۲۵۰ کلیدواژه خراب | MobileNetV3-**Large** + CLIP به‌عنوان پشتیبان |
| زمان طبقه‌بندی | هر پرس‌وجو | **یک بار، در ایندکس** |
| ترتیب مراحل | حیوان → غذا → چهره | **چهره → حیوان → غذا** |
| آستانه حیوان | ندارد | پسین رقابتی ≥ ۰.۵۵ |
| ثابت‌ها | دو نسخه‌ی واگرا | یک منبع + `validate()` |
| ذخیره‌سازی | `pickle` | `float32` خام |
| کلید جداول | ندارد | `PRIMARY KEY` |
| dtype | سه اصطلاح متناقض | `Runtime` یکپارچه |
| نتایج تکراری | ممکن | `best_per_image` |
| خطاها | `except:` خالی | چاپ و شمارش |

---

### 🔗 نسخه‌ی کامل با تست

این نوت‌بوک‌ها برای مطالعه و اجرای مستقیم‌اند. نسخه‌ی پکیج‌شده با
**۱۲۰ تست** و CI اینجاست:

`github.com/ParsaVictor/visual-intelligence-engine`

آنجا هر باگی که در این جدول می‌بینی، یک تست دارد که **بدون رفع، شکست
می‌خورد** — پس نمی‌تواند بی‌صدا برگردد.